# Notebook 04b — Looping an LLM over many documents

**Duration:** ~40 minutes

*This is an alternative version of notebook 04. It suits researchers whose data is a large collection of documents, such as web pages, transcripts or survey responses, that each need the same analysis.*

So far you have sent the LLM one piece of text at a time. Research doesn't always work like that. Imagine you have 500 interview transcripts, or every fundraising page on a website. You would not paste each one into a chat window. Instead, you write the instructions once and let Python repeat them for every document. This repetition is called a **loop**.

In this notebook you will:

1. Learn the **loop-and-collect** pattern on three short interview snippets
2. Load real fundraising campaigns from **Givealittle**, New Zealand's largest crowdfunding website
3. Loop an LLM over the campaigns to extract structured information from each one's text and photo
4. Ask the LLM to find themes across all the campaigns at once
5. Check the LLM's work, including against information it never saw

By the end you will have a pattern you can reuse on any collection of documents: **collect → loop → gather results → check**.

### A note on this data

These are real people's stories about illness, published on a public website. Public does not mean free of consequences. In this workshop we keep the data on your own computer, we do not share or publish anyone's name, and we discuss what an LLM should and should not be asked to judge about a person.

If you use this approach in your own research, you will need ethics approval, just as you would for any other data about people's health.

## Setup

Run this cell first. It loads your base URL and API key from the `.env` file you set up in notebook 01.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

import os, json, base64, requests, io
from dotenv import load_dotenv
from openai import OpenAI
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

# Load your base URL and API key from the .env file (set up in notebook 01)
load_dotenv(override=True)
if not os.getenv("LLM_BASE_URL") or not os.getenv("LLM_API_KEY"):
    raise RuntimeError("Could not find LLM_BASE_URL or LLM_API_KEY. Check your .env file (see notebook 01).")

client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Each Dell Pro Max machine serves one model. This is its name.
# It reads both text and images.
MODEL = "nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning-NVFP4"

# Turn off the model's "thinking" step. Answers come back several times faster,
# which matters when many people share one machine.
NO_THINKING = {"chat_template_kwargs": {"enable_thinking": False}}

print(f"Using model {MODEL}.")
print("Setup complete.")

This notebook needs a few extra tools. Run the cell below to load them.

- `pandas` makes tables of results, much like a spreadsheet. A pandas table is called a **DataFrame**.
- `tqdm` draws a progress bar, so you can see how far through a loop you are.

In [ ]:
# Extra tools for this notebook
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

print("Extra tools loaded.")

## Part 1 — What is a loop?

A **loop** tells Python: "for every item in this list, do these steps." A **list** is an ordered collection of items, written inside square brackets `[ ]`.

The cell below has a list of three made-up interview snippets. The loop prints each one with its word count. Notice that the steps are written once but run three times. Python knows which lines belong to the loop because they are **indented** (pushed in by four spaces).

In [ ]:
# A list of three made-up interview snippets
snippets = [
    "I waited eight months for surgery and lost my job in the meantime.",
    "The nurses were wonderful, but the hospital parking cost a fortune.",
    "We had to fly to Australia because the treatment isn't funded here.",
]

# The loop: for each snippet in the list, run the indented line below
for snippet in snippets:
    print(f"{len(snippet.split())} words: {snippet}")

Now the same loop with an LLM call inside. Each time round the loop, Python sends one snippet to the model and adds the answer to a results list.

This **loop-and-collect** pattern is the core of the notebook:

1. Make an empty list to hold the results.
2. Loop over your documents. For each one, call the LLM.
3. Add each answer to the results list.

Everything else in this notebook builds on these three steps.

In [ ]:
snippet_results = []  # an empty list; one answer per snippet ends up here

for snippet in snippets:
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a qualitative research assistant."},
            {"role": "user", "content": f"In five words or fewer, what is the main concern in this interview snippet?\n\n{snippet}"},
        ],
    )
    answer = response.choices[0].message.content.strip()
    snippet_results.append({"snippet": snippet, "main_concern": answer})
    print(f"{answer}  ←  {snippet}")

Three snippets took a few seconds. Three hundred would take longer, but you would not need to write a single extra line of code. That is the payoff of a loop.

## Part 2 — The data: Givealittle health campaigns

Our documents are **306 health-related fundraising campaigns** from Givealittle. Each campaign is a web page where someone tells a story about an illness or injury and asks the public for money.

### How the data was collected

We gathered these pages for you using **web scraping**: code that visits web pages and copies out the parts you care about, such as the title, the story and the amount raised. We did this once, ahead of time, and saved the results in a spreadsheet. That way, 40 people do not all visit the same website at the same moment.

If you scrape websites for your own research, be polite:

- **Check the site's `robots.txt` file.** Most websites publish a file at `/robots.txt` that says which pages automated programs may visit. Respect it.
- **Read the terms of use.** Some sites forbid scraping.
- **Say who you are, and slow down.** Identify your program and pause between requests so you do not overload the site.

### Loading the spreadsheet

The data is saved as a **CSV** file (comma-separated values, a plain-text table that Excel can open). Run the cell to load it into a DataFrame.

Unlike a folder of transcripts (one plain-text file per interview), this data is a table: one row per campaign. The free text we want to analyse (title, pitch, description, use of funds) sits alongside structured fields like the amount raised and the number of donors.

In [ ]:
campaigns = pd.read_csv("../data/givealittle/givealittle_health.csv")

print(f"Loaded {len(campaigns)} campaigns with {len(campaigns.columns)} columns:")
print(list(campaigns.columns))

### Look at one campaign first

Always look at your raw data before you analyse it. The cell below prints one campaign in full. Change `ROW` to look at another one. Python counts from 0, so the first campaign is row 0.

In [ ]:
ROW = 0  # change this number to look at a different campaign

campaign = campaigns.iloc[ROW]

print(f"Title:         {campaign['title']}")
print(f"Location:      {campaign['location']}")
print(f"Goal:          ${campaign['goal']:,.0f}")
print(f"Raised so far: ${campaign['amountRaised']:,.0f} from {campaign['n_donors']:.0f} donors")
print()
print("Description:")
print(campaign["description"])
print()
print("Use of funds:")
print(campaign["use_of_funds"])

And here is a quick view of the first ten rows, showing just a few columns.

In [ ]:
campaigns[["title", "location", "amountRaised", "goal", "n_donors"]].head(10)

### A helper for reading JSON

We will ask the model to answer in **JSON** (a structured text format you met in notebook 03). Models sometimes wrap their JSON in extra markers like ` ```json `. This small function strips them off before Python reads the JSON. It is the same clean-up step you used in earlier notebooks, packaged so the loop can reuse it.

In [ ]:
def parse_json(raw_text):
    """Turn the model's reply into Python data, removing ```json fences if present."""
    cleaned = raw_text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
        if cleaned.startswith("json"):
            cleaned = cleaned[4:].strip()
    return json.loads(cleaned)

## Part 3 — The loop: Extract structured information from text and photos

Now the main job. For each campaign we send the LLM three things:

1. the instructions (the **extraction prompt**),
2. the campaign's main photo, which Givealittle calls the "hero" image,
3. the campaign text.

The model returns one JSON object per campaign, with about 35 fields.

Some fields use **ICD-10**, the International Classification of Diseases (10th revision). It is the World Health Organization's standard list of codes for health conditions. Its **chapters** group conditions into broad families. For example, cancers fall under "Neoplasms".

**Read the prompt before you run it.** Some fields ask for facts that are in the text, such as the condition or how the money will be used. Others ask the model to judge truthfulness, deservingness or attractiveness, or to guess ethnicity from a photo. We come back to these in Part 5. For now, as you read, ask yourself: *which of these could a careful human coder answer from this page, and which could nobody answer reliably?*

In [ ]:
#### The extraction prompt
# The same prompt works for every campaign, so we define it once as a constant.
# It asks for one JSON object per campaign - some fields come from the text,
# some (like smiling or image_type) can only come from the image.
extraction_prompt = """
The below message is text extracted from givealittle, a crowdfunding platform. It's a health related campaign.
I've also included the hero image for the campaign.
For the text below, extract the following information, in JSON format:
condition: the primary health condition mentioned in the text
ICD10: the ICD10 code for the primary health condition
ICD: the top level ICD chapter for the primary health condition. One of:
    Chapter A00-B99 - Certain infectious and parasitic diseases
    Chapter C00-D49 - Neoplasms
    Chapter D50-D89 - Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism
    Chapter E00-E89 - Endocrine, nutritional and metabolic diseases
    Chapter F01-F99 - Mental, Behavioral and Neurodevelopmental disorders
    Chapter G00-G99 - Diseases of the nervous system
    Chapter H00-H59 - Diseases of the eye and adnexa
    Chapter H60-H95 - Diseases of the ear and mastoid process
    Chapter I00-I99 - Diseases of the circulatory system
    Chapter J00-J99 - Diseases of the respiratory system
    Chapter K00-K95 - Diseases of the digestive system
    Chapter L00-L99 - Diseases of the skin and subcutaneous tissue
    Chapter M00-M99 - Diseases of the musculoskeletal system and connective tissue
    Chapter N00-N99 - Diseases of the genitourinary system
    Chapter O00-O9A - Pregnancy, childbirth and the puerperium
    Chapter P00-P96 - Certain conditions originating in the perinatal period
    Chapter Q00-Q99 - Congenital malformations, deformations and chromosomal abnormalities
    Chapter R00-R99 - Symptoms, signs and abnormal clinical and laboratory findings, not elsewhere classified
    Chapter S00-T88 - Injury, poisoning and certain other consequences of external causes
    Chapter V00-Y99 - External causes of morbidity
    Chapter Z00-Z99 - Factors influencing health status and contact with health services
    Chapter U00-U85 - Codes for special purposes
name: the name of the person this campaign is for
gender: the gender of the person this campaign is for, one of Male, Female or Other/unknown
age: the age of the person this campaign is for
age_group: the age group of the person this campaign is for, one of 0-14, 15-64, 65+ or indeterminate/unknown
ethnicity: the ethnicity of the person this campaign is for. If not mentioned in the text, guess their ethnicity from the image.
urgency: a number from 0-100, indicating how urgent the need is
sentiment: a number from 0-100, indicating the sentiment of the text, where 100 is the most positive, and 0 is the most negative
truth: a number from 0-100, indicating how truthful the text is, where 100 is the most truthful, and 0 is the least truthful
notes: any additional information about how you processed this text, such as warnings or errors
smiling: a boolean indicating whether the person in the image is smiling
deservingness: a number from 0-100, indicating how deserving the person is of receiving funds, where 100 is the most deserving, and 0 is the least deserving
attractiveness: a number from 0-100, indicating how attractive the person is, where 100 is the most attractive, and 0 is the least attractive
use: The main use of the raised funds - one or more (comma separated) of: medical expenses, experimental therapies, travel expenses, lost wages
region: The region in New Zealand where the person is located, one of: Northland, Auckland, Waikato, Bay of Plenty, Gisborne, Hawke's Bay, Taranaki, Manuwatū-Whanganui, Wellington, Tasman, Nelson, Marlborough, West Coast, Canterbury, Otago, Southland
narrative_clarity: a number from 0-100, indicating how clear the narrative is, where 100 is the most clear, and 0 is the least clear
narrative_quality: a number from 0-100, indicating how well written the narrative is
emotional_tone: grateful | desperate | hopeful | neutral | etc
image_type: selfie | portrait | symbolic | environment | group | other
face_visible: true | false
facial_expression: smiling | neutral | serious | emotional | not_detectable
image_quality: high | medium | low
progression: a number from 0-100, indicating how advanced the condition is, where 100 is the most advanced, and 0 is the least advanced
treatment: a number from 0-100, indicating how much treatment the person has received, where 100 is the most treatment, and 0 is the least treatment
treatment_effectiveness: a number from 0-100, indicating how effective the treatment has been, where 100 is the most effective, and 0 is the least effective
treatment_side_effects: a number from 0-100, indicating how severe the side effects of the treatment have been
site: If the campaign is for cancer, what is the primary cancer site?
stage: If the campaign is for cancer, what stage is the cancer at?
reason: Summarise how donated funds will be used and the reason for requesting donations

Do not include comments in your JSON response. Only respond with the JSON object. Make sure the JSON is valid.
"""

### Preparing the photos

This is the same image preparation as notebook 05:

1. Download the image.
2. Make sure it is no more than 400 pixels wide. Smaller images are faster to process on a shared machine.
3. Convert it to **base64**, a way of writing an image as plain text so it can travel inside the request to the model.

The cell defines a function for these steps and tests it on the first campaign's photo.

In [ ]:
# Identify ourselves when downloading images, as notebook 05 does
IMAGE_HEADERS = {"User-Agent": "UoA-eResearch-LLM-Workshop/1.0 (researchdata@auckland.ac.nz)"}


def image_to_base64(url, width=400):
    """Download an image, shrink it to at most `width` pixels wide, and return it as base64 text."""
    response = requests.get(url, headers=IMAGE_HEADERS, timeout=30)
    img = Image.open(io.BytesIO(response.content)).convert("RGB")
    if img.width > width:
        height = int(img.height * width / img.width)
        img = img.resize((width, height))

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


# Test it on the first campaign's photo
test_image = image_to_base64(campaigns.loc[0, "hero"])
print(campaigns.loc[0, "title"])
display(IPImage(data=base64.b64decode(test_image), format="jpeg"))

### The extraction loop

This is the loop from the Givealittle example, adapted for the workshop machines. Things to notice:

- **`content` is a list of parts** (text, image, text) rather than one string, just as in notebook 05.
- **We do not send the campaign's location.** The spreadsheet records where each campaign is based. We keep that back on purpose, so we can use it to check the model in Part 5.
- **`MAX_CAMPAIGNS` caps the loop.** Each campaign can take several seconds, and everyone in the room shares the same machine. After the workshop you can raise the cap and run all 306.
- **We take a random sample**, not just the first rows. The first rows of a spreadsheet are often not typical: they may be the newest or the most popular. `random_state=42` makes the sample the same every time you run it, so your results can be reproduced. Change the number to draw a different sample.
- **There are three `try` / `except` blocks**: one for the photo download, one for the LLM call, and one for broken JSON. If anything goes wrong for one campaign, the loop notes the problem and moves on.

Run the cell, then read the next section while it works.

In [ ]:
MAX_CAMPAIGNS = 10  # raise this when you are not sharing the machine

to_process = campaigns.sample(n=MAX_CAMPAIGNS, random_state=42)
extracted = []  # one extracted record per campaign ends up here

for row in tqdm(to_process.itertuples(), total=len(to_process)):

    # The free text sits in several fields, so stitch them into one labelled block.
    # Location is left out on purpose (see Part 5).
    campaign_text = "\n\n".join([
        f"Title: {row.title}",
        f"Pitch: {row.pitch}",
        f"Description: {row.description}",
        f"Use of funds: {row.use_of_funds}",
    ])

    try:
        image_base64 = image_to_base64(row.hero)
    except Exception as error:
        print(f"{row.uri}: could not download the photo ({error})")
        continue

    messages = [
        {
            "role": "user",
            "content": [
                # Part 1: the instructions
                {"type": "text", "text": extraction_prompt},
                # Part 2: the hero image
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}},
                # Part 3: the campaign text itself
                {"type": "text", "text": campaign_text},
            ]
        }
    ]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            extra_body=NO_THINKING,
            temperature=0,  # extraction: we want reproducible output
            messages=messages,
        )
    except Exception as error:
        print(f"{row.uri}: LLM call failed ({error})")
        continue

    raw_output = response.choices[0].message.content

    try:
        result = parse_json(raw_output)
    except json.JSONDecodeError:
        print(f"{row.uri}: model did not return valid JSON. Raw output:\n{raw_output}\n")
        continue

    # Keep the campaign address so results can be joined back to the original data
    result["uri"] = row.uri
    extracted.append(result)

extracted_df = pd.DataFrame(extracted)

os.makedirs("../data/extracted", exist_ok=True)
extracted_df.to_csv("../data/extracted/givealittle_extracted.csv", index=False)

print(f"Extracted {len(extracted_df)} of {len(to_process)} campaigns.")
print("Saved to ../data/extracted/givealittle_extracted.csv")
extracted_df

### Looking across the results

Each row of `extracted_df` is one campaign, and each column is one field from the prompt. Now we can count. The cell below counts the most common values in three columns and draws a bar chart of health-condition chapters.

With ten campaigns these counts are anecdotes, not findings. But the same code works unchanged on 1,000 campaigns. That is when counts become a dataset.

In [ ]:
for column in ["ICD", "use", "emotional_tone"]:
    if column in extracted_df:
        print(f"--- {column} ---")
        print(extracted_df[column].astype(str).value_counts().to_string())
        print()

# Bar chart of ICD chapters
if "ICD" in extracted_df:
    counts = extracted_df["ICD"].astype(str).value_counts()
    fig, ax = plt.subplots(figsize=(10, 4))
    counts.sort_values().plot.barh(ax=ax, color="#4A90D9")
    ax.set_xlabel("Number of campaigns")
    ax.set_title("Health-condition chapter (ICD-10), as coded by the LLM")
    plt.tight_layout()
    plt.show()

## Part 4 — Finding themes across campaigns

So far, each LLM call has seen one campaign. To find **themes** (recurring patterns of meaning across many documents), the model needs to see many campaigns at once.

We do this in two steps:

1. **Loop:** summarise each document. You have already done this. The `reason` field from Part 3 is a one-line summary of why each campaign is asking for money.
2. **Combine:** send all the summaries to the model in a single call and ask for themes.

Why not send every full page at once? Ten pages might fit. Five hundred would not: every model has a limit on how much text it can read in one go. Short summaries let you analyse far more documents. This is the same idea as the "semantic enrichment" step in notebook 04, where an LLM produced compact summaries before any themes were found.

In [ ]:
# Number each summary so the model can tell us which campaigns fit each theme
summary_lines = []
for number, reason in enumerate(extracted_df["reason"].fillna(""), start=1):
    summary_lines.append(f"Campaign {number}: {reason}")
summaries_text = "\n".join(summary_lines)

theme_response = client.chat.completions.create(
    model=MODEL,
    extra_body=NO_THINKING,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a qualitative research assistant analysing New Zealand crowdfunding campaigns."
        },
        {
            "role": "user",
            "content": f"""Below are short summaries of {len(summary_lines)} health-related crowdfunding campaigns from New Zealand.
Each summary says how donated funds will be used and why the campaign is asking for help.

Identify 3 to 5 recurring themes in the reasons people give for asking the public for money.

For each theme, provide:
- A short theme name
- A one-sentence description
- 5 to 10 keywords
- The numbers of the campaigns that show this theme

Return ONLY a JSON array:
[
  {{
    "theme": "Theme name",
    "description": "One-sentence description",
    "keywords": ["keyword1", "keyword2"],
    "campaigns": [1, 3]
  }}
]

Summaries:
{summaries_text}"""
        }
    ]
)

themes = parse_json(theme_response.choices[0].message.content)

print(f"{len(themes)} themes identified")
print("=" * 60)
for i, theme in enumerate(themes, 1):
    print(f"\n{i}. {theme['theme']}")
    print(f"   {theme['description']}")
    print(f"   Keywords:  {', '.join(theme['keywords'])}")
    print(f"   Campaigns: {theme.get('campaigns')}")

Some questions to consider:

- Do the themes match your own sense of the campaigns? Would you have found similar ones?
- Pick one theme and look at the campaign numbers listed for it (`summary_lines` shows which is which). Does each campaign really belong there?
- Is any campaign left out of every theme? Why might that be?
- The model only saw the one-line summaries, not the full pages. What might have been lost in that step?

In hand-coded thematic analysis you would go back to the data to test a theme. The next part does exactly that.

In [ ]:
# Show the numbered summaries, so you can check the campaign numbers above
print(summaries_text)

## Part 5 — Checking the LLM's work

Running a loop is easy. Trusting what comes out of it is the hard part. Here are three checks you can use on any LLM loop.

### Check 1 — Read one campaign yourself (a spot check)

A **spot check** means picking a few results and checking them by hand against the original. The cell below prints one campaign's original text next to what the LLM extracted.

For each field, decide: correct, wrong, or *not in the text at all*? Change `CHECK` to look at another campaign.

In [ ]:
CHECK = 0  # change this number to check a different campaign

result = extracted_df.iloc[CHECK]
original = campaigns[campaigns["uri"] == result["uri"]].iloc[0]

print("ORIGINAL PAGE")
print("=" * 60)
print(f"Title: {original['title']}\n")
print(original["description"])
print(f"\nUse of funds: {original['use_of_funds']}")

print("\n\nWHAT THE LLM EXTRACTED")
print("=" * 60)
for field in ["condition", "ICD10", "ICD", "name", "gender", "age", "age_group", "use", "site", "stage", "reason"]:
    print(f"{field:12s} {result.get(field)}")

Watch for answers that are plausible but unsupported. For example, an exact age when the page never gives one, or a cancer stage when none is mentioned. When the model gives a confident answer that has no basis in the source, that is a **hallucination**. In a loop over hundreds of documents, nobody reads every answer. So hallucinations get through unless you go looking for them.

### Check 2 — Compare against information the model never saw

The spreadsheet records where each campaign is based. We kept that location out of the text we sent to the model. The prompt still asked the model for a `region`. So we can compare the model's answer with the real one.

This is a small version of the validation idea in notebook 04: compare the LLM's output against an **independent reference**, information the model could not have copied.

In [ ]:
# Join the LLM's answers back onto the original data using the campaign address
region_check = extracted_df[["uri", "region"]].merge(
    campaigns[["uri", "title", "location"]], on="uri"
)

# Locations look like "Christchurch, Canterbury" or just "Auckland". The region is the last part.
region_check["page_region"] = region_check["location"].fillna("").str.split(",").str[-1].str.strip()
region_check["match"] = (
    region_check["page_region"].str.lower() == region_check["region"].astype(str).str.lower()
)

print(f"Agreement: {region_check['match'].sum()} of {len(region_check)} campaigns ({region_check['match'].mean():.0%})")
region_check[["title", "location", "region", "match"]]

Look at the mismatches. Each one usually has one of three causes:

- **A spelling difference**, such as "Manawatu-Wanganui" in the spreadsheet against "Manuwatū-Whanganui" in the prompt. That is a data-cleaning problem, not a model problem.
- **The location is not a region at all**, such as "Nationwide".
- **The text never says where the person lives.** Look back at the prompt: it gives the model a list of regions but no "unknown" option. So when the text is silent, the model has to pick one anyway, and it guesses.

That second cause is a general lesson. **If a prompt forces a choice, you will get a choice, whether or not the evidence supports it.** Giving the model a way to say "not stated" is one of the simplest ways to reduce hallucination.

### Check 3 — Fields that nobody could verify

Some fields in the prompt ask the model to judge a person rather than read the text. Run the cell to see what it said.

In [ ]:
judgement_fields = ["ethnicity", "smiling", "truth", "deservingness", "attractiveness", "urgency"]
extracted_df[["name"] + [f for f in judgement_fields if f in extracted_df]]

Discuss with your neighbour:

- **What could the model be basing an "attractiveness" score on?** Whose idea of attractiveness has it learned, and from what data?
- **What does "truth" mean here?** The model cannot check any facts. What might a low truth score actually be picking up: writing style, spelling, how dramatic the story is?
- **Ethnicity guessed from a photo.** Suppose you used this field to ask whether some groups raise less money than others. The guesses would not be random noise. They would be **biased**: wrong more often for some groups than others. That bias flows straight into your findings.
- **A number like 73 looks precise.** Precision is not accuracy. A 0 to 100 scale invites you to average it and plot it as if it were a measurement.
- **For a real study, which fields would you keep, which would you drop, and which would you give to a human coder?**

There are no single right answers. But you should be able to defend each field in your methods section, and to an ethics committee.

## Optional stretch — Reuse the pattern on your own documents

The loop does not care where the documents come from. Suppose you have a folder of interview transcripts saved as `.txt` files. Only the first two lines of the loop change:

```python
from pathlib import Path

transcript_results = []
for path in Path("../data/my_transcripts").glob("*.txt"):   # every .txt file in the folder
    text = path.read_text()
    response = client.chat.completions.create(
        model=MODEL,
        extra_body=NO_THINKING,
        temperature=0,
        messages=[{"role": "user", "content": your_prompt + "\n\n" + text}],
    )
    transcript_results.append({"file": path.name, "answer": response.choices[0].message.content})
```

If you have time, try one of these:

- Write your own `extraction_prompt` with five fields that matter for *your* research, and re-run the extraction loop.
- Add `"not stated"` as an option for `region` in the prompt, re-run the extraction loop, and see whether the agreement in Check 2 changes.
- Run the extraction loop twice and compare the two `extracted_df` tables. Which fields stay the same, and which change?

## What you accomplished

In this notebook you:

- Learned the **loop-and-collect** pattern: write the steps once, run them for every document, and gather the results in a table
- Saw how web-scraped data arrives as a table, and why you look at the raw data before analysing it
- Looped an LLM over a sample of campaigns to **extract** structured data from text and photos together
- Used per-document summaries to find **themes** across the whole collection
- **Checked** the output three ways, and saw how a prompt that forces a choice invites hallucination

Your validation toolkit now includes:

| Technique | What it checks | When you used it |
|-----------|---------------|------------------|
| **Consistency check** | Does the method give the same answer twice? | Notebook 02 (temperature) |
| **Cross-method comparison** | Do two methods agree? | Notebook 02 |
| **Spot check** | Is each extracted field supported by the source? | This notebook (Check 1) |
| **Independent reference** | Does the output match information the model never saw? | This notebook (Check 2) |
| **Field critique** | Should the model be asked this at all? | This notebook (Check 3) |

The key insight: a loop multiplies whatever your prompt does, good or bad. One shaky field in one campaign is a small problem. The same shaky field across 10,000 campaigns becomes a published finding. So check before you scale up.

**Next up:** Notebook 05 looks more closely at extracting information from images. The data type changes. The methodology does not.